In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    xgb = None

import os
NOTEBOOK_DIR = Path().resolve()
if NOTEBOOK_DIR.name == 'notebooks':
    BASE_DIR = NOTEBOOK_DIR.parent
else:
    BASE_DIR = NOTEBOOK_DIR

DATA_PATH = BASE_DIR / "data" / "your_datasets" / "diabetes_data.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path(r"d:/sugarcare_backend/data/your_datasets/diabetes_data.csv")

DIABETES_DATASET_PATH = BASE_DIR / "data" / "your_datasets" / "diabetes_dataset.csv"
if not DIABETES_DATASET_PATH.exists():
    DIABETES_DATASET_PATH = Path(r"d:/sugarcare_backend/data/your_datasets/diabetes_dataset.csv")

BRFSS_PATHS = [
    BASE_DIR / "data" / "your_datasets" / "diabetes_012_health_indicators_BRFSS2015.csv",
    BASE_DIR / "data" / "your_datasets" / "diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    BASE_DIR / "data" / "your_datasets" / "diabetes_binary_health_indicators_BRFSS2015.csv",
]

if not any(p.exists() for p in BRFSS_PATHS):
    BRFSS_PATHS = [
        Path(r"d:/sugarcare_backend/data/your_datasets/diabetes_012_health_indicators_BRFSS2015.csv"),
        Path(r"d:/sugarcare_backend/data/your_datasets/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"),
        Path(r"d:/sugarcare_backend/data/your_datasets/diabetes_binary_health_indicators_BRFSS2015.csv"),
    ]

OUTPUT_DIR = BASE_DIR / "models"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path(r"d:/sugarcare_backend/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Data path: {DATA_PATH} (exists: {DATA_PATH.exists()})")
print(f"Output directory: {OUTPUT_DIR}")


def build_population_stats(paths: List[Path]) -> Dict[float, Dict[str, float]]:
    frames = []
    for path in paths:
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        df.columns = [c.strip() for c in df.columns]
        if 'BMI' not in df.columns:
            continue
        df['BMI'] = pd.to_numeric(df['BMI'], errors='coerce')
        if 'Diabetes_binary' not in df.columns:
            if 'Diabetes_012' in df.columns:
                df['Diabetes_binary'] = pd.to_numeric(df['Diabetes_012'], errors='coerce')
                df['Diabetes_binary'] = df['Diabetes_binary'].apply(lambda x: 1.0 if x >= 1 else 0.0)
            else:
                df['Diabetes_binary'] = np.nan
        else:
            df['Diabetes_binary'] = pd.to_numeric(df['Diabetes_binary'], errors='coerce')
        for col in ['HighBP', 'HighChol']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            else:
                df[col] = np.nan
        frames.append(df[['BMI', 'Diabetes_binary', 'HighBP', 'HighChol']])
    if not frames:
        return {}
    merged = pd.concat(frames, ignore_index=True)
    merged = merged.dropna(subset=['BMI'])
    stats = merged.groupby('BMI').agg({
        'Diabetes_binary': 'mean',
        'HighBP': 'mean',
        'HighChol': 'mean',
    }).dropna()
    population_stats: Dict[float, Dict[str, float]] = {}
    for bmi_value, row in stats.iterrows():
        population_stats[float(bmi_value)] = {
            'diabetes_risk': float(row['Diabetes_binary']),
            'high_bp_rate': float(row['HighBP']),
            'high_chol_rate': float(row['HighChol']),
        }
    return population_stats


population_stats = build_population_stats(BRFSS_PATHS)
print(f"Population BMI buckets loaded: {len(population_stats)}")

MEAL_DATA_PATH = BASE_DIR / "data" / "your_datasets" / "meal_recommender.csv"
if not MEAL_DATA_PATH.exists():
    MEAL_DATA_PATH = Path(r"d:/sugarcare_backend/data/your_datasets/meal_recommender.csv")

meal_nutrition_map: Dict[str, Dict[str, float]] = {}
if MEAL_DATA_PATH.exists():
    try:
        mdf = pd.read_csv(MEAL_DATA_PATH)
        name_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["meal", "food", "name", "item", "dish"])]
        gi_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["glycemic_index", "gi"])]
        carbs_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["carbs", "carbohydrates"])]
        fiber_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["fiber", "fibre"])]
        protein_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["protein", "proteins"])]
        fat_cols = [c for c in mdf.columns if any(kw in c.lower() for kw in ["fat", "fats"])]
        
        if name_cols:
            name_col = name_cols[0]
            for _, row in mdf.iterrows():
                meal_name = str(row[name_col]).strip().lower()
                gi = float(pd.to_numeric(row.get(gi_cols[0] if gi_cols else None, 50), errors='coerce') or 50)
                carbs = float(pd.to_numeric(row.get(carbs_cols[0] if carbs_cols else None, 0), errors='coerce') or 0)
                fiber = float(pd.to_numeric(row.get(fiber_cols[0] if fiber_cols else None, 0), errors='coerce') or 0)
                protein = float(pd.to_numeric(row.get(protein_cols[0] if protein_cols else None, 0), errors='coerce') or 0)
                fat = float(pd.to_numeric(row.get(fat_cols[0] if fat_cols else None, 0), errors='coerce') or 0)
                
                meal_nutrition_map[meal_name] = {
                    'gi': gi,
                    'carbs': carbs,
                    'fiber': fiber,
                    'protein': protein,
                    'fat': fat
                }
        print(f"Loaded nutrition data for {len(meal_nutrition_map)} meals")
    except Exception as e:
        print(f"Warning: Could not load meal nutrition data: {e}")

def compute_absorption_curve_features(carbs_g: float, fiber_g: float, gi: float, protein_g: float = 0, fat_g: float = 0) -> Tuple[float, float, float]:
    """
    Simplified absorption model based on Arleth two-compartment model.
    Computes glucose absorption curve and extracts 3 key parameters:
    - p1: time to peak [minutes]
    - p2: time to 50% of peak [minutes]
    - p3: rate of absorption at maximum [g/minute]
    """
    if carbs_g <= 0:
        return (30.0, 15.0, 0.1)
    
    CHOAvail = 0.76
    available_carbs = carbs_g * CHOAvail
    
    fiber_factor = max(0.5, 1.0 - (fiber_g / (carbs_g + 1.0)) * 0.3)
    fat_protein_delay = min(20.0, (fat_g + protein_g) * 0.5)
     
    gi_factor = gi / 100.0
    base_absorption_rate = available_carbs * gi_factor * 0.02
    
    time_to_peak = 30.0 + fat_protein_delay - (fiber_factor * 10.0)
    time_to_peak = max(15.0, min(90.0, time_to_peak))
    
    time_to_half_peak = time_to_peak * 0.4
    time_to_half_peak = max(5.0, min(45.0, time_to_half_peak))
    
    max_absorption_rate = base_absorption_rate * (1.0 + (1.0 - fiber_factor))
    max_absorption_rate = max(0.05, min(2.0, max_absorption_rate))
    
    return (time_to_peak, time_to_half_peak, max_absorption_rate)

_df = pd.read_csv(DATA_PATH)
_df['date'] = pd.to_datetime(_df['date'])
_df = _df.sort_values(['user_id', 'date']).reset_index(drop=True)

for col in ['physical_activity', 'diet', 'medication_adherence', 'stress_level', 'sleep_hours', 'hydration_level', 'bmi']:
    if col in _df.columns:
        _df[col] = pd.to_numeric(_df[col], errors='coerce').fillna(_df[col].median())

if DIABETES_DATASET_PATH.exists():
    try:
        _df_enhanced = pd.read_csv(DIABETES_DATASET_PATH)
        
        activity_map = {'Low': 10.0, 'Moderate': 30.0, 'High': 50.0}
        if 'Physical_Activity_Level' in _df_enhanced.columns:
            _df_enhanced['physical_activity_numeric'] = _df_enhanced['Physical_Activity_Level'].map(activity_map).fillna(30.0)
        
        enhanced_rows = []
        np.random.seed(42)
        rng = np.random.default_rng(42)
        
        for idx, row in _df_enhanced.iterrows():
            fasting_glucose = float(row.get('Fasting_Blood_Glucose', 120.0))
            hba1c = float(row.get('HbA1c', 7.0))
            bmi_val = float(row.get('BMI', 25.0))
            activity_numeric = float(row.get('physical_activity_numeric', 30.0))
            calories = float(row.get('Dietary_Intake_Calories', 2000.0))
            
            if calories < 1500:
                diet_quality = 0.3
            elif calories < 2500:
                diet_quality = 0.6
            else:
                diet_quality = 0.9
            
            num_readings = rng.integers(5, 11)
            base_date = pd.Timestamp('2023-01-01') + pd.Timedelta(days=rng.integers(0, 365))
            
            glucose_mean = fasting_glucose
            glucose_std = (hba1c - 5.0) * 10.0 + 15.0
            
            for day_offset in range(num_readings):
                date_val = base_date + pd.Timedelta(days=day_offset)
                
                trend_factor = np.sin(day_offset * 0.3) * 10.0
                noise = rng.normal(0, glucose_std * 0.3)
                glucose = glucose_mean + trend_factor + noise
                glucose = max(70.0, min(300.0, glucose))
                
                stress = float(rng.uniform(0.5, 2.0))
                sleep = float(rng.uniform(5.0, 9.0))
                hydration = float(rng.uniform(0.8, 1.2))
                medication = float(rng.uniform(0.7, 1.0))
                
                enhanced_rows.append({
                    'user_id': f"enhanced_{idx}",
                    'date': date_val,
                    'blood_glucose': glucose,
                    'physical_activity': activity_numeric,
                    'diet': diet_quality,
                    'medication_adherence': medication,
                    'stress_level': stress,
                    'sleep_hours': sleep,
                    'hydration_level': hydration,
                    'bmi': bmi_val,
                    'hba1c': hba1c,
                    'calories': calories
                })
        
        if enhanced_rows:
            _df_enhanced_ts = pd.DataFrame(enhanced_rows)
            _df_enhanced_ts = _df_enhanced_ts.sort_values(['user_id', 'date']).reset_index(drop=True)
            
            _df = pd.concat([_df, _df_enhanced_ts], ignore_index=True)
            _df = _df.sort_values(['user_id', 'date']).reset_index(drop=True)
            print(f"Enhanced dataset: Added {len(enhanced_rows)} synthetic readings from {len(_df_enhanced)} patients")
    except Exception as e:
        print(f"Warning: Could not load enhanced diabetes dataset: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"Enhanced dataset not found at {DIABETES_DATASET_PATH}")


def lookup_population_stats(bmi_value: float) -> Dict[str, float]:
    if not population_stats or pd.isna(bmi_value):
        return {'diabetes_risk': 0.35, 'high_bp_rate': 0.35, 'high_chol_rate': 0.30}
    keys = np.array(list(population_stats.keys()), dtype=float)
    idx = int(np.argmin(np.abs(keys - float(bmi_value))))
    key = float(keys[idx])
    return population_stats.get(key, {'diabetes_risk': 0.35, 'high_bp_rate': 0.35, 'high_chol_rate': 0.30})


rows = []
_df_sorted = _df.sort_values(['user_id', 'date']).reset_index(drop=True)

for i in range(1, len(_df_sorted)):
    start = max(0, i - 7)
    recent_values = _df_sorted.loc[start:i-1, 'blood_glucose'].to_numpy()
    if recent_values.size == 0:
        continue
    recent_times = _df_sorted.loc[start:i-1, 'date']
    current_time = _df_sorted.loc[i, 'date']
    last_time = recent_times.iloc[-1]

    time_since_last_hours = (current_time - last_time).total_seconds() / 3600.0
    time_since_last_hours = float(max(0.0, time_since_last_hours))
    if len(recent_times) > 1:
        intervals = recent_times.diff().dropna().dt.total_seconds() / 3600.0
        avg_interval_hours = float(max(0.0, intervals.mean()))
    else:
        avg_interval_hours = time_since_last_hours
    last_reading_hour = float(last_time.hour + last_time.minute / 60.0)
    reading_count = float(len(recent_times))

    current_target = float(_df_sorted.loc[i, 'blood_glucose'])
    avg_glucose = float(np.mean(recent_values))
    glucose_variability = float(np.std(recent_values))
    slope = float(np.polyfit(np.arange(len(recent_values)), recent_values, 1)[0]) if len(recent_values) >= 2 else 0.0
    current_prev = float(recent_values[-1])
    min_glucose = float(np.min(recent_values))
    max_glucose = float(np.max(recent_values))
    pa = float(_df_sorted.loc[i, 'physical_activity']) if 'physical_activity' in _df_sorted.columns else 0.0
    diet_val = float(_df_sorted.loc[i, 'diet']) if 'diet' in _df_sorted.columns else 0.0
    med = float(_df_sorted.loc[i, 'medication_adherence']) if 'medication_adherence' in _df_sorted.columns else 0.0
    stress = float(_df_sorted.loc[i, 'stress_level']) if 'stress_level' in _df_sorted.columns else 0.0
    sleep = float(_df_sorted.loc[i, 'sleep_hours']) if 'sleep_hours' in _df_sorted.columns else 0.0
    hydr = float(_df_sorted.loc[i, 'hydration_level']) if 'hydration_level' in _df_sorted.columns else 0.0
    bmi = float(_df_sorted.loc[i, 'bmi']) if 'bmi' in _df_sorted.columns else 0.0
    
    hba1c = float(_df_sorted.loc[i, 'hba1c']) if 'hba1c' in _df_sorted.columns else None
    if hba1c is None or pd.isna(hba1c):
        hba1c = (avg_glucose + 46.7) / 28.7
    hba1c = float(hba1c)
    calories = float(_df_sorted.loc[i, 'calories']) if 'calories' in _df_sorted.columns else None
    
    if diet_val == 1.0:
        estimated_carbs = 30.0
        estimated_fiber = 8.0
        estimated_gi = 45.0
        estimated_protein = 20.0
        estimated_fat = 8.0
    elif diet_val == 0.5:
        estimated_carbs = 50.0
        estimated_fiber = 4.0
        estimated_gi = 60.0
        estimated_protein = 15.0
        estimated_fat = 12.0
    else:
        estimated_carbs = 70.0
        estimated_fiber = 2.0
        estimated_gi = 75.0
        estimated_protein = 10.0
        estimated_fat = 15.0
    
    if calories is not None and not pd.isna(calories):
        calorie_factor = calories / 2000.0
        estimated_carbs *= calorie_factor
        estimated_protein *= calorie_factor
        estimated_fat *= calorie_factor
    
    abs_p1, abs_p2, abs_p3 = compute_absorption_curve_features(
        estimated_carbs, estimated_fiber, estimated_gi, estimated_protein, estimated_fat
    )
    
    meal_impact = (abs_p3 * 20.0) - 15.0
    
    if pa < 10:
        activity_impact = 25.0
    elif pa < 20:
        activity_impact = 15.0
    elif pa < 30:
        activity_impact = -5.0
    elif pa < 50:
        activity_impact = -15.0
    else:
        activity_impact = -25.0 
        
    if len(recent_values) >= 3:
        recent_slopes = np.diff(recent_values)
        acceleration = float(np.mean(np.diff(recent_slopes))) if len(recent_slopes) > 1 else 0.0
    else:
        acceleration = 0.0
    
    glucose_range = max_glucose - min_glucose
    
    if len(recent_values) >= 3:
        trend_strength = float(np.abs(slope) / (glucose_variability + 1.0))
    else:
        trend_strength = 0.0
    
    is_morning = 6 <= last_time.hour < 12
    is_afternoon = 12 <= last_time.hour < 18
    is_evening = 18 <= last_time.hour < 24
    is_night = 0 <= last_time.hour < 6

    pop_stats = lookup_population_stats(bmi)

    rows.append({
        'avg_glucose': avg_glucose,
        'trend': slope,
        'glucose_variability': glucose_variability,
        'current_glucose': current_prev,
        'min_glucose': min_glucose,
        'max_glucose': max_glucose,
        'glucose_range': glucose_range,
        'acceleration': acceleration,
        'trend_strength': trend_strength,
        'meal_impact': meal_impact,
        'absorption_time_to_peak': abs_p1,
        'absorption_time_to_half_peak': abs_p2,
        'absorption_max_rate': abs_p3,
        'activity_impact': activity_impact,
        'medication_adherence': med,
        'stress_level': stress,
        'sleep_hours': sleep,
        'hydration_level': hydr,
        'bmi': bmi,
        'hba1c': hba1c,
        'time_since_last_reading_hours': time_since_last_hours,
        'avg_interval_hours': avg_interval_hours,
        'last_reading_hour': last_reading_hour,
        'reading_count': reading_count,
        'is_morning': 1.0 if is_morning else 0.0,
        'is_afternoon': 1.0 if is_afternoon else 0.0,
        'is_evening': 1.0 if is_evening else 0.0,
        'is_night': 1.0 if is_night else 0.0,
        'population_diabetes_risk': pop_stats['diabetes_risk'],
        'population_high_bp_rate': pop_stats['high_bp_rate'],
        'population_high_chol_rate': pop_stats['high_chol_rate'],
        'target_glucose': current_target,
    })

df = pd.DataFrame(rows)
df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
print('Samples:', len(df))
if df.empty:
    raise RuntimeError('No training samples were generated. Please verify the input CSV columns and data.')

FEATURES = [
    'avg_glucose', 'trend', 'glucose_variability', 'current_glucose',
    'min_glucose', 'max_glucose', 'glucose_range', 'acceleration', 'trend_strength',
    'meal_impact', 'absorption_time_to_peak', 'absorption_time_to_half_peak', 'absorption_max_rate',
    'activity_impact',
    'medication_adherence', 'stress_level', 'sleep_hours', 'hydration_level', 'bmi', 'hba1c',
    'time_since_last_reading_hours', 'avg_interval_hours', 'last_reading_hour',
    'reading_count', 'is_morning', 'is_afternoon', 'is_evening', 'is_night',
    'population_diabetes_risk', 'population_high_bp_rate', 'population_high_chol_rate',
]

missing_features = [f for f in FEATURES if f not in df.columns]
if missing_features:
    print(f"Warning: Missing features {missing_features}, removing from feature list")
    FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES].values
y = df['target_glucose'].values

y_mean, y_std = np.mean(y), np.std(y)
outlier_mask = np.abs(y - y_mean) <= 3 * y_std
X = X[outlier_mask]
y = y[outlier_mask]
print(f"After outlier removal: {len(X)} samples (removed {len(df) - len(X)})")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=None)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining with {len(FEATURES)} features on {len(X_train)} samples")
print(f"Feature names: {FEATURES}\n")

candidates = {
    'gbr': GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1, 
        subsample=0.8, min_samples_split=10, min_samples_leaf=5,
        random_state=42, verbose=0
    ), 
    'rf': RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_split=10, 
        min_samples_leaf=4, max_features='sqrt',
        random_state=42, n_jobs=-1, verbose=0
    ),
    'etr': ExtraTreesRegressor(
        n_estimators=300, max_depth=12, min_samples_split=10,
        min_samples_leaf=4, max_features='sqrt',
        random_state=42, n_jobs=-1, verbose=0
    ),
    'ridge': Ridge(alpha=10.0, random_state=42),
    'elastic': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42, max_iter=2000),
}

if XGB_AVAILABLE:
    candidates['xgboost'] = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.08, 
        subsample=0.85, colsample_bytree=0.85, min_child_weight=3,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, tree_method='hist', verbosity=0
    )

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}
cv_r2_scores = {}

print("CROSS-VALIDATION RESULTS")
print("=" * 60)

for name, model in candidates.items():
    mae_scores = -cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_scores[name] = float(np.mean(mae_scores))
    
    r2_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, 
                                scoring='r2', n_jobs=-1)
    cv_r2_scores[name] = float(np.mean(r2_scores))
    
    print(f"{name:12s} | CV MAE: {cv_scores[name]:6.2f} | CV R2: {cv_r2_scores[name]:6.3f}")

best_name = min(cv_scores, key=lambda k: (cv_scores[k], -cv_r2_scores[k]))
best_model = candidates[best_name]

print(f"\n{'='*60}")
print(f"SELECTED MODEL: {best_name}")
print(f"{'='*60}")

best_model.fit(X_train_scaled, y_train)

pred = best_model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"Test MAE:  {mae:.2f} mg/dL")
print(f"Test RMSE: {rmse:.2f} mg/dL")
print(f"Test R2:   {r2:.3f}")

within_10 = np.sum(np.abs(pred - y_test) <= 10) / len(y_test) * 100
within_20 = np.sum(np.abs(pred - y_test) <= 20) / len(y_test) * 100
within_30 = np.sum(np.abs(pred - y_test) <= 30) / len(y_test) * 100

print(f"\nPrediction Accuracy:")
print(f"  Within ±10 mg/dL: {within_10:.1f}%")
print(f"  Within ±20 mg/dL: {within_20:.1f}%")
print(f"  Within ±30 mg/dL: {within_30:.1f}%")

meal_map: dict = {}
if meal_nutrition_map:
    for meal_name, nutrition in meal_nutrition_map.items():
        carbs = nutrition.get('carbs', 0)
        fiber = nutrition.get('fiber', 0)
        gi = nutrition.get('gi', 50)
        protein = nutrition.get('protein', 0)
        fat = nutrition.get('fat', 0)
        
        abs_p1, abs_p2, abs_p3 = compute_absorption_curve_features(carbs, fiber, gi, protein, fat)
        meal_map[meal_name] = {
            'absorption_time_to_peak': abs_p1,
            'absorption_time_to_half_peak': abs_p2,
            'absorption_max_rate': abs_p3,
            'gi': gi,
            'carbs': carbs
        }
    print(f"Built absorption model features for {len(meal_map)} meals")
else:
    print("No meal nutrition data available for absorption model")

export = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': FEATURES,
    'model_name': best_name,
    'cv_mae': cv_scores[best_name],
    'cv_r2': cv_r2_scores[best_name],
    'test_mae': mae,
    'test_rmse': rmse,
    'test_r2': r2,
    'test_accuracy_within_10': within_10,
    'test_accuracy_within_20': within_20,
    'test_accuracy_within_30': within_30,
    'meal_map': meal_map,
    'meal_nutrition_map': meal_nutrition_map,
    'population_stats': population_stats,
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'feature_count': len(FEATURES),
    'absorption_model_available': True,
}

export['features'] = FEATURES

model_path = OUTPUT_DIR / 'sugar_forecast_trained.pkl'
joblib.dump(export, model_path)
print(f'\n✅ Model saved: {model_path}')
print(f'   Model: {best_name}')
print(f'   Features: {len(FEATURES)}')
print(f'   Test MAE: {mae:.2f} mg/dL')
print(f'   Test R2: {r2:.3f}')

if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': FEATURES,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(f'\nTop 10 Most Important Features:')
    print(importance_df.head(10).to_string(index=False))
